In [ ]:
# import necessary libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# load the data
data=pd.read_csv('movies.csv')
data.head()

In [82]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [83]:
# we have not missing values in the data

In [84]:
data.nunique()

movieId    9742
title      9737
genres      951
dtype: int64

In [85]:
# check for dupiicates
data[data['title'].duplicated(keep = False)].sort_values(by = 'title')

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi


In [86]:
# some movies have same titles with different movieid's so we keep first
data.drop_duplicates(subset='title', keep='first', inplace=True, ignore_index=True)

In [87]:
# remove brackets from title
data['title']=data.title.str.replace('(','').str.replace(')','').str.strip()
data.head()

,movieId,title,genres
0,1,Toy Story 1995,Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji 1995,Adventure|Children|Fantasy
2,3,Grumpier Old Men 1995,Comedy|Romance
3,4,Waiting to Exhale 1995,Comedy|Drama|Romance
4,5,Father of the Bride Part II 1995,Comedy


In [88]:
# create year column
data['year']=data['title'].apply(lambda x: x[-4:])
         # or 
#data.title.str[-4:]
data.head()

,movieId,title,genres,year
0,1,Toy Story 1995,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji 1995,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men 1995,Comedy|Romance,1995
3,4,Waiting to Exhale 1995,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II 1995,Comedy,1995


In [89]:
data.year.unique()

array(['1995', '1994', '1996', '1976', '1992', '1967', '1993', '1964',
       '1977', '1965', '1982', '1990', '1991', '1989', '1937', '1940',
       '1969', '1981', '1973', '1970', '1955', '1959', '1968', '1988',
       '1997', '1972', '1943', '1952', '1951', '1957', '1961', '1958',
       '1954', '1934', '1944', '1960', '1963', '1942', '1941', '1953',
       '1939', '1950', '1946', '1945', '1938', '1947', '1935', '1936',
       '1956', '1949', '1932', '1975', '1974', '1971', '1979', '1987',
       '1986', '1980', '1978', '1985', '1966', '1962', '1983', '1984',
       '1948', '1933', '1931', '1922', '1998', '1929', '1930', '1927',
       '1928', '1999', '2000', '1926', '1919', '1921', '1925', '1923',
       '2001', '2002', '2003', '1920', '1915', '1924', '2004', '1916',
       '1917', '2005', '2006', '1902', 'on 5', '1903', '2007', '2008',
       '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016',
       '2017', '2018', '1908', ' One', 'Road', 'tson', 'mals', 'rson',
      

In [90]:
# find titles that did not have year
data[data.year.str.isdigit()==False]

,movieId,title,genres,year
6058,40697,Babylon 5,Sci-Fi,on 5
9029,140956,Ready Player One,Action|Sci-Fi|Thriller,One
9089,143410,Hyena Road,(no genres listed),Road
9134,147250,The Adventures of Sherlock Holmes and Doctor W...,(no genres listed),tson
9175,149334,Nocturnal Animals,Drama|Thriller,mals
9255,156605,Paterson,(no genres listed),rson
9363,162414,Moonlight,Drama,ight
9444,167570,The OA,(no genres listed),e OA
9509,171495,Cosmos,(no genres listed),smos
9510,171631,Maria Bamford: Old Baby,(no genres listed),Baby


In [91]:
#there are only few rows that did not contain years so we will drop that rows
data=data[data.year.str.isdigit()==True] 
data.head()

,movieId,title,genres,year
0,1,Toy Story 1995,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji 1995,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men 1995,Comedy|Romance,1995
3,4,Waiting to Exhale 1995,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II 1995,Comedy,1995


In [92]:
# convert type of years to numeric
data['year']=data['year'].astype('int64')

In [93]:
print(sorted(data.year.unique()))

[1902, 1903, 1908, 1915, 1916, 1917, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]


In [94]:
# In general trend of movies change in every 10-15 years so we will keep data accordingly
data[(data['year']>= 1985) & (data['year']<=2018)].shape

(7714, 4)

In [95]:
data = data[(data['year']>= 1985) & (data['year']<=2018)]

In [96]:
# remove years from title
data['title']=data['title'].apply(lambda x:x[:-5])
data.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [97]:
# here problem in the data that i used for this project that some movie names ended with (, The and , A and , An)  that should be prefix but they become suffix
data[data['title'] == 'Indian in the Cupboard, The']

,movieId,title,genres,year
53,60,"Indian in the Cupboard, The",Adventure|Children|Fantasy,1995


In [98]:
# we have 1576 words that has mistake of wrong title name
data[data['title'].str.endswith((', The', ', A', ', An'))]['title'].count()

1113

In [99]:
def fix_title(title):
    if title.endswith(", The"): 
        return "The " + title[:-5]   
    elif title.endswith(", A"):
        return "A " + title[:-3]
    elif title.endswith(", An"):
        return "An " + title[:-4]
    return title

In [100]:
data['title']=data['title'].apply(fix_title)

In [101]:
data[data['title'].str.endswith((', The', ', A', ', An'))]['title'].count()

0

In [102]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7714 entries, 0 to 9736
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  7714 non-null   int64 
 1   title    7714 non-null   object
 2   genres   7714 non-null   object
 3   year     7714 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 301.3+ KB


In [103]:
# now find unique genres  and apply one hot encoding to get features that are genres 
genres_list=list(data.genres.unique())
genres_list[:5]

['Adventure|Animation|Children|Comedy|Fantasy',
 'Adventure|Children|Fantasy',
 'Comedy|Romance',
 'Comedy|Drama|Romance',
 'Comedy']

In [104]:
unique_genres=set()
for i in genres_list:
    unique_genres.update(i.split('|'))
     
print(sorted(unique_genres))

['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [105]:
#convert unique geners to list because set is not mutable 
unique_genres=list(unique_genres)
type(unique_genres)

list

In [106]:
   # or  
unique_genres=[]
 
for i in genres_list:
    li=i.split('|')
    
    for j in li: 
        if j not in unique_genres:
            unique_genres.append(j)
print(unique_genres)

# we use it because set is not mutable

['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy', 'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror', 'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX', 'Western', 'Film-Noir', '(no genres listed)']


In [107]:
data[data.genres=='(no genres listed)'].shape

(22, 4)

In [108]:
# we have very less data that have no genres listed so we drop them 
df=data[data.genres!='(no genres listed)'] 
df.reset_index(drop=True,inplace=True)

In [109]:
df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [110]:
unique_genres.remove('(no genres listed)')
print(unique_genres)

['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy', 'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror', 'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX', 'Western', 'Film-Noir']


In [111]:
#this is like a base to fill the genre values for final_df so let go ahed  
genres_df=pd.get_dummies(unique_genres).astype('int64')
genres_df.head()

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [112]:
genres_df=genres_df[genres_df.index==0]
genres_df        

# this step is mostly used.

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [113]:
genres_df['Adventure']=0
genres_df

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [114]:
df=pd.concat([df,genres_df],axis=1)
df.head()

,movieId,title,genres,year,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,Adventure|Children|Fantasy,1995,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Grumpier Old Men,Comedy|Romance,1995,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Father of the Bride Part II,Comedy,1995,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [115]:
df.fillna(0,inplace=True)
df.head()       
# we do that stuff to get all values 0 so we will add new values to it 

,movieId,title,genres,year,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,Adventure|Children|Fantasy,1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men,Comedy|Romance,1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II,Comedy,1995,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


####  keep attention on next line 

In [116]:
# before apply it we have to understand the next cell 

def create_encoding(tmp_df):
    genre_l = tmp_df['genres']   # generally it returns a series but we will apply the function row wise it 
    #print(genre_l)
    genre_l = genre_l.split('|')   # it give a list of genres for each row
    #print(genre_l)
    for g in genre_l:
        tmp_df[g]=1            # # it assign value 1 (indicating this movie belongs to that genre)
        #display(tmp_df)
    return tmp_df 

In [117]:
final_df = df.apply(lambda x: create_encoding(x),axis=1)     # axis=1 means apply function to each row of genre column.
final_df.head()

,movieId,title,genres,year,Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995,0.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,Adventure|Children|Fantasy,1995,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men,Comedy|Romance,1995,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II,Comedy,1995,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [118]:
final_df.columns

Index(['movieId', 'title', 'genres', 'year', 'Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery',
       'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'],
      dtype='object')

In [119]:
# prepare data to pass to cosine similarity for getting similarity between movies 

movie_df=final_df[['movieId','year', 'Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery',
       'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']]
movie_df.head()

,movieId,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1995,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,1995,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,1995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,4,1995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,5,1995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [120]:
movie_df.index = movie_df['movieId'].values  

In [121]:
movie_df.drop(columns=['movieId'],inplace=True)

In [122]:
movie_df  # it includes all the based on which we find similarity between movies

,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,1995,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1995,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,1995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,1995,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,2017,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,2017,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,2017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193587,2018,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [123]:
# getting the similarities betweeen movies, here data is at row level(each row represents a single movie similarity with ohter movies),  as high the the values as similar the movie
movie_similarity=cosine_similarity(movie_df,movie_df)
movie_similarity=pd.DataFrame(movie_similarity)
movie_similarity

,0,1,2,3,4,5,6,7,8,9,...,7682,7683,7684,7685,7686,7687,7688,7689,7690,7691
0,1.000000,1.000000,0.999999,0.999999,0.999999,0.999999,0.999999,1.000000,0.999999,0.999999,...,0.999999,0.999999,0.999999,0.999999,0.999999,1.000000,1.0,0.999999,0.999999,0.999999
1,1.000000,1.000000,0.999999,0.999999,0.999999,0.999999,0.999999,1.000000,0.999999,0.999999,...,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999,1.0,1.000000,0.999999,0.999999
2,0.999999,0.999999,1.000000,1.000000,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
3,0.999999,0.999999,1.000000,1.000000,1.000000,0.999999,1.000000,0.999999,0.999999,0.999999,...,0.999999,1.000000,1.000000,0.999999,0.999999,0.999999,1.0,1.000000,0.999999,1.000000
4,0.999999,0.999999,1.000000,1.000000,1.000000,0.999999,1.000000,1.000000,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7687,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,0.999999,1.000000,1.0,0.999999,1.000000,1.000000
7688,1.000000,1.000000,1.000000,1.000000,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
7689,0.999999,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.999999,1.000000,1.000000,1.000000,1.000000,0.999999,1.0,1.000000,1.000000,1.000000
7690,0.999999,0.999999,1.000000,0.999999,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000


In [124]:
# to make the index relavant that show the true relation as data is at row level where each row represent one single movie's similarity with others 
movie_similarity.index=movie_df.index.values
movie_similarity.columns=movie_df.index.values
movie_similarity

,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
1,1.000000,1.000000,0.999999,0.999999,0.999999,0.999999,0.999999,1.000000,0.999999,0.999999,...,0.999999,0.999999,0.999999,0.999999,0.999999,1.000000,1.0,0.999999,0.999999,0.999999
2,1.000000,1.000000,0.999999,0.999999,0.999999,0.999999,0.999999,1.000000,0.999999,0.999999,...,0.999999,0.999999,0.999999,0.999999,0.999999,0.999999,1.0,1.000000,0.999999,0.999999
3,0.999999,0.999999,1.000000,1.000000,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
4,0.999999,0.999999,1.000000,1.000000,1.000000,0.999999,1.000000,0.999999,0.999999,0.999999,...,0.999999,1.000000,1.000000,0.999999,0.999999,0.999999,1.0,1.000000,0.999999,1.000000
5,0.999999,0.999999,1.000000,1.000000,1.000000,0.999999,1.000000,1.000000,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,0.999999,1.000000,1.0,0.999999,1.000000,1.000000
193583,1.000000,1.000000,1.000000,1.000000,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000
193585,0.999999,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.999999,1.000000,1.000000,1.000000,1.000000,0.999999,1.0,1.000000,1.000000,1.000000
193587,0.999999,0.999999,1.000000,0.999999,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.000000


In [125]:
# now index of moive_df and movie_similarity df is same and index of final df is different          argsort

In [135]:
# udf to get top similar movies when input is movie_id

def recommend(movie_id):
    sim_score = movie_similarity.loc[movie_id]  # finding similarity scores of entered movie 
    sim_score = sim_score.sort_values(ascending = False)  
    sim_score = sim_score.drop(movie_id)    # remove same movie
    top_5_sim = sim_score.head(5).index   # top 5 similarity movie_id's
    
    for i in top_5_sim:
        print(final_df[final_df['movieId'] == i]['title'].values[0])    # get the names of top 5 simlar movies 

In [136]:
recommend(1)

Antz
Toy Story 2
The Emperor's New Groove
The Adventures of Rocky and Bullwinkle
Monsters, Inc.


In [ ]:
# udf to get top similar movies when input is movie_title

def recommend_mov(movie_title): 
    movie_id = final_df[final_df['title'] == movie_title]['movieId'].values[0]
    sim_scores = movie_similarity.loc[movie_id]   # Series with movieId index
    sim_scores = sim_scores.sort_values(ascending=False)
    sim_scores = sim_scores.drop(movie_id)   # remove same movie
    top_movies = sim_scores.head(5).index   # directly movieIds

    for i in top_movies:
        print(final_df[final_df['movieId'] == i]['title'].values[0])

In [ ]:
recommend_mov('Jumanji')

### we have to pass these to streamlit :
     data_dict: preprocessed data in dictionary form to get data 
     udf or model : we create same there 
     movie_similarity : to get similarities between movies

In [ ]:
# for deploy the model
import pickle

In [ ]:
pickle.dump(final_df.to_dict(), open('data_dict.pkl','wb'))

In [ ]:
pickle.dump(movie_similarity, open('similarity.pkl','wb'))